In [1]:
import traci
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
SUMO_BINARY = "sumo"
SUMO_CONFIG = "simulation.sumocfg"
SIM_END = 4500
CONTROL_STEP = 15

NUM_LANES = 4

In [3]:
RAMPS = {
    "Ramp 1": {
        "loc": (1, 1),
        "x_range": (1400, 3600),
        "q0": 10356,
        "detectors": {
            "up":  {"ids": [f"det_a1_up_{i}"  for i in range(4)], "shift": 0},
            "down1": {"ids": [f"det_a1_dn1_{i}" for i in range(4)],   "shift": 30},
            "down2": {"ids": [f"det_a1_dn2_{i}" for i in range(4)],   "shift": 58},
        },
    },
    "Ramp 2": {
        "loc": (0, 1),
        "x_range": (2100, 4600),
        "q0": 9432,
        "detectors": {
            "up": {"ids": [f"det_a2_up_{i}"  for i in range(4)], "shift": 0},
            "down1": {"ids": [f"det_a2_dn1_{i}" for i in range(1, 5)], "shift": 29},
            "down2": {"ids": [f"det_a2_dn2_{i}" for i in range(4)],   "shift": 51},
        },
    },
    "Ramp 3": {
        "loc": (1, 0),
        "x_range": (1500, 3900),
        "q0": 8472,
        "detectors": {
            "up": {"ids": [f"det_a3_up_{i}"  for i in range(4)]
                        , "shift": 0},
            "down1": {"ids": [f"det_a3_dn1_{i}" for i in range(4)],   "shift": 22},
            "down2": {"ids": [f"det_a3_dn2_{i}" for i in range(1, 5)], "shift": 43},
        },
    },
    "Ramp 4": {
        "loc": (0, 0),
        "x_range": (1000, 3900),
        "q0": 6204,
        "detectors": {
            "up": {"ids": [f"det_a4_up_{i}"  for i in range(4)], "shift": 0},
            "down1": {"ids": [f"det_a4_dn1_{i}" for i in range(4)],   "shift": 39},
            "down2": {"ids": [f"det_a4_dn2_{i}" for i in range(4)],   "shift": 62},
        },
    },
}

In [4]:
# ----------- SIMULATION -----------

history = {
    "Ramp 1": {
            "up": [], "down1": [], "down2": []
        },
    "Ramp 2": {
            "up": [], "down1": [], "down2": []
        },
    "Ramp 3": {
            "up": [], "down1": [], "down2": []
        },
    "Ramp 4": {
            "up": [], "down1": [], "down2": []
        },
}

In [5]:
def getAllVehCounts(history):
    for ramp_name, item in RAMPS.items():
        up_ids = item['detectors']['up']['ids']
        down1_ids = item['detectors']['down1']['ids']
        down2_ids = item['detectors']['down2']['ids']

        up_count = np.sum([traci.inductionloop.getLastStepVehicleNumber(d) for d in up_ids])/4
        down1_count = np.sum([traci.inductionloop.getLastStepVehicleNumber(d) for d in down1_ids])/4
        down2_count = np.sum([traci.inductionloop.getLastStepVehicleNumber(d) for d in down2_ids])/4

        history[ramp_name]['up'].append(up_count)
        history[ramp_name]['down1'].append(down1_count)
        history[ramp_name]['down2'].append(down2_count)

    return history

In [6]:
routes = {
    1:  ["ramp1_to_end", "ramp1_to_off2", "ramp1_to_off3", "ramp1_to_off4"],
    2:  ["ramp2_to_end", "ramp2_to_off3", "ramp2_to_off4"],
    3:  ["ramp3_to_end", "ramp3_to_off4"],
    4:  ["ramp4_to_end"]
}

probs = {
    1:  [0.7, 0.1, 0.1, 0.1],
    2:  [0.8, 0.1, 0.1],
    3:  [0.9, 0.1],
    4:  [1.0]
}

In [7]:
ramp_times = [0] + list(np.arange(600, 3601, step=300)) + [SIM_END]

In [8]:
def get_ramp1_flow(t, ramp_demands):
    return np.interp(t, ramp_times, ramp_demands[1])

def get_ramp2_flow(t, ramp_demands):
    return np.interp(t, ramp_times, ramp_demands[2])

def get_ramp3_flow(t, ramp_demands):
    return np.interp(t, ramp_times, ramp_demands[3])

def get_ramp4_flow(t, ramp_demands):
    return np.interp(t, ramp_times, ramp_demands[4])

In [ ]:
def insertRampVehicles(rampIndex, t, ramp_demands):
    fncs = {1: get_ramp1_flow, 2: get_ramp2_flow, 3: get_ramp3_flow, 4: get_ramp4_flow}

    get_flow_fnc = fncs[rampIndex]

    V = get_flow_fnc(t, ramp_demands)
    p = V / (3600 * 2)

    for lane in range(2):
        if np.random.random() < p:
            route = np.random.choice(routes[rampIndex], p=probs[rampIndex])

            traci.vehicle.add(f"ramp{rampIndex}_{t}_{lane}", route, typeID="car_ramp",
                departLane="best", departPos="free", departSpeed="random")

In [10]:
def insertVehicles(ramp_demands):
    t = traci.simulation.getTime()

    for i in range(4):
        insertRampVehicles(i+1, t, ramp_demands)

In [11]:
def getFlow(counts):
    counts = np.asarray(counts, dtype=float)

    flow = np.zeros(len(counts))
    half = FLOW_WINDOW // 2

    for i in range(len(counts)):
        s = max(0, i - half)
        e = min(len(counts), i + half + 1)

        window_sum = np.sum(counts[s:e])
        window_length = e - s

        # Convert vehicles/second to vehicles/hour
        flow[i] = (window_sum / window_length) * 3600

    return flow*4

FLOW_WINDOW = 300

In [12]:
def plotFlow(times, b):
    ramp1_flow = getFlow(history['Ramp 1']['down1'])
    ramp2_flow = getFlow(history['Ramp 2']['down1'])
    ramp3_flow = getFlow(history['Ramp 3']['down1'])
    ramp4_flow = getFlow(history['Ramp 4']['down1'])

    times_to_plot = np.arange(1200, 3601, step=300)
    x = np.asarray(times)[times_to_plot]

    y_ramp1 = ramp1_flow[x]
    y_ramp2 = ramp2_flow[x]
    y_ramp3 = ramp3_flow[x]
    y_ramp4 = ramp4_flow[x]

    plt.plot(x, y_ramp1, color="g", marker=".", label="Ramp 1")
    plt.plot(x, y_ramp2, color="b", marker="^", label="Ramp 2")
    plt.plot(x, y_ramp3, color="orange", marker="*", label="Ramp 3")
    plt.plot(x, y_ramp4, color="r", marker="x", label="Ramp 4")

    plt.legend()
    plt.savefig(os.path.join("plot", f"flow_with_bonus{b}"))
    plt.close()

In [13]:
bonuses = [0, 100, 200, 300, 400, 500]

for b in bonuses:
    history = {
        "Ramp 1": {
                "up": [], "down1": [], "down2": []
            },
        "Ramp 2": {
                "up": [], "down1": [], "down2": []
            },
        "Ramp 3": {
                "up": [], "down1": [], "down2": []
            },
        "Ramp 4": {
                "up": [], "down1": [], "down2": []
            },
    }

    ramp_demands = {
        1:  [1700, 1700, 2175, 2175, 2175, 2175, 2175, 2175, 1400, 1400, 1400, 1400, 1400],
        2:  [1030, 1030, 1243, 1243, 1243, 1243, 1243, 1080, 960, 960, 960, 960, 960],
        3:  [1103, 1103, 1292, 1292, 1292, 1292, 1292, 1103, 1086, 1086, 1086, 1046, 1046],
        4:  [1313, 1313, 2221, 2221, 2221, 2221, 2221, 1313, 1300, 1300, 1275, 1250, 1250]
    }

    ramp_demands[1] = np.asarray(ramp_demands[1])
    ramp_demands[2] = np.asarray(ramp_demands[2]) + b * 1
    ramp_demands[3] = np.asarray(ramp_demands[3]) + b * 2
    ramp_demands[4] = np.asarray(ramp_demands[4]) + b * 3

    traci.start([SUMO_BINARY, "-c", SUMO_CONFIG, "--no-step-log", "true", "--no-warnings"])

    times = []

    for i in tqdm(range(SIM_END)):
        insertVehicles(ramp_demands)

        history = getAllVehCounts(history)

        times.append(i)

        traci.simulationStep()

    traci.close()

    plotFlow(times, b)

 Retrying in 1 seconds
***Starting server on port 63086 ***
Loading net-file from 'network.net.xml' ... done (8ms).
Loading additional-files from 'detectors.add.xml' ... done (3ms).
Loading route-files incrementally from 'routes.rou.xml'
Loading done.
Simulation version 1.26.0 started with time: 0.00.


  0%|          | 0/4500 [00:00<?, ?it/s]


TypeError: VehicleDomain.add() got an unexpected keyword argument 'insertionChecks'

# Figure 12

In [ ]:
# Define the edges that make up each physical ramp
RAMP_EDGES = {
    "Ramp 1": ["edge_ramp_2_1", "edge_ramp_1_1"],
    "Ramp 2": ["edge_ramp_2_2", "edge_ramp_1_2"],
    "Ramp 3": ["edge_ramp_2_3", "edge_ramp_1_3"],
    "Ramp 4": ["edge_ramp_2_4", "edge_ramp_1_4"],
}

# Setup history trackers
queue_history = {"Ramp 1": [], "Ramp 2": [], "Ramp 3": [], "Ramp 4": []}
pending_history = []
plot_times = []

ramp_demands = {
    1:  [1700, 1700, 2175, 2175, 2175, 2175, 2175, 2175, 1400, 1400, 1400, 1400, 1400],
    2:  [1030, 1030, 1243, 1243, 1243, 1243, 1243, 1080, 960, 960, 960, 960, 960],
    3:  [1103, 1103, 1292, 1292, 1292, 1292, 1292, 1103, 1086, 1086, 1086, 1046, 1046],
    4:  [1313, 1313, 2221, 2221, 2221, 2221, 2221, 1313, 1300, 1300, 1275, 1250, 1250]
}

# --- YOUR SIMULATION LOOP ---
for i in tqdm(range(SIM_END)):

    insertVehicles(ramp_demands)

    traci.simulationStep()

    # 1. Track the physical queue length on each ramp
    for ramp_name, edges in RAMP_EDGES.items():
        # Sum the vehicles currently occupying the two edges of the ramp
        q_length = sum(traci.edge.getLastStepVehicleNumber(edge_id) for edge_id in edges)
        queue_history[ramp_name].append(q_length)

    # 2. Track the invisible "Delayed" vehicles that SUMO couldn't spawn
    pending_history.append(len(traci.simulation.getPendingVehicles()))
    plot_times.append(i)

traci.close()